# SQL Practice Problems — Runnable Notebook

This notebook creates a small retail dataset in SQLite and demonstrates representative answers across core SQL topics. Try editing the queries and testing edge cases.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript('''
CREATE TABLE customers (
 customer_id INTEGER PRIMARY KEY,
 customer_name TEXT NOT NULL,
 city TEXT
);
CREATE TABLE products (
 product_id INTEGER PRIMARY KEY,
 product_name TEXT NOT NULL,
 category TEXT NOT NULL,
 price REAL NOT NULL CHECK(price >= 0)
);
CREATE TABLE orders (
 order_id INTEGER PRIMARY KEY,
 customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
 order_date TEXT NOT NULL,
 status TEXT NOT NULL
);
CREATE TABLE order_items (
 order_id INTEGER NOT NULL REFERENCES orders(order_id),
 product_id INTEGER NOT NULL REFERENCES products(product_id),
 quantity INTEGER NOT NULL CHECK(quantity > 0),
 unit_price REAL NOT NULL CHECK(unit_price >= 0),
 PRIMARY KEY(order_id, product_id)
);
''')
conn.executemany("INSERT INTO customers VALUES (?, ?, ?)", [
 (1, "Asha Sharma", "Indore"), (2, "Ravi Patel", "Bhopal"),
 (3, "Neha Singh", "Indore"), (4, "Mohan Das", None)
])
conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", [
 (10, "Notebook", "Stationery", 50), (11, "Pen Pack", "Stationery", 120),
 (12, "Backpack", "Accessories", 850), (13, "Water Bottle", "Accessories", 250),
 (14, "Desk Lamp", "Home", 650)
])
conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", [
 (100, 1, "2026-01-03", "delivered"), (101, 1, "2026-01-15", "shipped"),
 (102, 2, "2026-02-02", "processing"), (103, 3, "2026-02-20", "delivered")
])
conn.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", [
 (100, 10, 2, 50), (100, 11, 1, 120), (101, 12, 1, 850),
 (102, 13, 2, 250), (103, 14, 1, 650), (103, 10, 3, 50)
])
print("Retail practice database created.")


Retail practice database created.


## 1. Inspect the data

In [2]:
for table in ["customers", "products", "orders", "order_items"]:
    print("\\n" + table)
    display(pd.read_sql_query(f"SELECT * FROM {table}", conn))


\ncustomers


,customer_id,customer_name,city
0,1,Asha Sharma,Indore
1,2,Ravi Patel,Bhopal
2,3,Neha Singh,Indore
3,4,Mohan Das,None


\nproducts


,product_id,product_name,category,price
0,10,Notebook,Stationery,50.0
1,11,Pen Pack,Stationery,120.0
2,12,Backpack,Accessories,850.0
3,13,Water Bottle,Accessories,250.0
4,14,Desk Lamp,Home,650.0


\norders


,order_id,customer_id,order_date,status
0,100,1,2026-01-03,delivered
1,101,1,2026-01-15,shipped
2,102,2,2026-02-02,processing
3,103,3,2026-02-20,delivered


\norder_items


,order_id,product_id,quantity,unit_price
0,100,10,2,50.0
1,100,11,1,120.0
2,101,12,1,850.0
3,102,13,2,250.0
4,103,14,1,650.0
5,103,10,3,50.0


## 2. Filtering, sorting, and limiting

In [3]:
pd.read_sql_query("SELECT product_name, price FROM products WHERE price > 100 ORDER BY price DESC LIMIT 3", conn)


,product_name,price
0,Backpack,850.0
1,Desk Lamp,650.0
2,Water Bottle,250.0


## 3. Join orders to customers

In [4]:
pd.read_sql_query('''
SELECT o.order_id, c.customer_name, o.order_date, o.status
FROM orders o JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_date, o.order_id
''', conn)


,order_id,customer_name,order_date,status
0,100,Asha Sharma,2026-01-03,delivered
1,101,Asha Sharma,2026-01-15,shipped
2,102,Ravi Patel,2026-02-02,processing
3,103,Neha Singh,2026-02-20,delivered


## 4. Aggregation and HAVING

In [5]:
pd.read_sql_query('''
SELECT c.customer_name, COUNT(DISTINCT o.order_id) AS order_count,
       COALESCE(SUM(oi.quantity * oi.unit_price), 0) AS spending
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
LEFT JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY c.customer_id, c.customer_name
ORDER BY spending DESC
''', conn)


,customer_name,order_count,spending
0,Asha Sharma,2,1070.0
1,Neha Singh,1,800.0
2,Ravi Patel,1,500.0
3,Mohan Das,0,0.0


## 5. Products never sold

In [6]:
pd.read_sql_query('''
SELECT p.product_id, p.product_name
FROM products p LEFT JOIN order_items oi ON oi.product_id = p.product_id
WHERE oi.product_id IS NULL
''', conn)


,product_id,product_name


## 6. CTE for order revenue

In [7]:
pd.read_sql_query('''
WITH order_revenue AS (
 SELECT order_id, SUM(quantity * unit_price) AS revenue
 FROM order_items GROUP BY order_id
)
SELECT order_id, revenue FROM order_revenue
ORDER BY revenue DESC
''', conn)


,order_id,revenue
0,101,850.0
1,103,800.0
2,102,500.0
3,100,220.0


## 7. Ranking with a window function

In [8]:
pd.read_sql_query('''
SELECT product_name, price,
       RANK() OVER (ORDER BY price DESC) AS price_rank
FROM products
ORDER BY price_rank, product_name
''', conn)


,product_name,price,price_rank
0,Backpack,850.0,1
1,Desk Lamp,650.0,2
2,Water Bottle,250.0,3
3,Pen Pack,120.0,4
4,Notebook,50.0,5


## 8. Previous order per customer

In [9]:
pd.read_sql_query('''
SELECT customer_id, order_id, order_date,
       LAG(order_date) OVER (
           PARTITION BY customer_id ORDER BY order_date, order_id
       ) AS previous_order_date
FROM orders
ORDER BY customer_id, order_date
''', conn)


,customer_id,order_id,order_date,previous_order_date
0,1,100,2026-01-03,None
1,1,101,2026-01-15,2026-01-03
2,2,102,2026-02-02,None
3,3,103,2026-02-20,None


## 9. Monthly order counts

In [10]:
pd.read_sql_query('''
SELECT substr(order_date, 1, 7) AS order_month, COUNT(*) AS order_count
FROM orders GROUP BY substr(order_date, 1, 7)
ORDER BY order_month
''', conn)


,order_month,order_count
0,2026-01,2
1,2026-02,2


## 10. Query plan and index

In [11]:
conn.execute("CREATE INDEX idx_orders_customer_date ON orders(customer_id, order_date)")
pd.read_sql_query("EXPLAIN QUERY PLAN SELECT * FROM orders WHERE customer_id = 1 AND order_date >= '2026-01-01' AND order_date < '2026-02-01'", conn)


,id,parent,notused,detail
0,3,0,0,SEARCH orders USING INDEX idx_orders_customer_...


## Practice tip\n\nChange the sample data, add customers without orders, introduce duplicate prices, and test NULL values. Good SQL should handle these cases deliberately.